# EcoSort: NEW plastic-wrapper training
Select a T4 GPU, run the code cell, and upload **EcoSort_TrashNet_split.zip** once. The notebook downloads new TACO packaging images and trains a fresh six-class classifier. No old model is required. Keep the runtime connected until the result ZIP downloads.

Actual usable image counts are printed before training. Test images are held out. Wrappers map to `plastic` under the project packaging convention; this is not verified material composition. Training has not run merely by opening this notebook.

Source and detailed method: https://github.com/pedropro/TACO. TACO: Proença and Simões (2020).

In [ ]:
# NEW wrapper training. Upload only your original six-class TrashNet split ZIP.
from pathlib import Path
import importlib.metadata
import json, shutil, subprocess, sys, traceback
import torch
from google.colab import files
from IPython.display import display, Image as DisplayImage

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("Upload ONLY EcoSort_TrashNet_split.zip. No old model or three-photo folder is needed.")
uploaded=files.upload()
if len(uploaded)!=1 or not next(iter(uploaded)).lower().endswith('.zip'):
    raise ValueError("Upload exactly one TrashNet split ZIP.")
archive=Path(next(iter(uploaded))).resolve()
del uploaded
work=Path('/content/ecosort_taco_work')
work.mkdir(parents=True,exist_ok=True)
source=work/'Src'
source.mkdir(exist_ok=True)

MODULES={'taco_data': '"""TACO plastic-packaging crops, with source-grouped splits and provenance."""\nimport concurrent.futures\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image, ImageDraw, ImageOps\n\nTACO_COMMIT = \'70479ad9adc3da971a8ae75ac97eb1026910eec7\'\nANNOTATIONS_URL = f\'https://raw.githubusercontent.com/pedropro/TACO/{TACO_COMMIT}/data/annotations.json\'\nCATEGORIES = {\'Plastic film\', \'Other plastic wrapper\', \'Crisp packet\'}\nCLASSES = [\'cardboard\', \'glass\', \'metal\', \'paper\', \'plastic\', \'trash\']\nPOLICY = (\'EcoSort plastic includes visually identified flexible plastic packaging, including \'\n          \'metallized snack packets, following the project packaging convention. \'\n          \'Source labels do not verify chemical composition or recyclability.\')\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef pixel_hash(image):\n    rgb = image.convert(\'RGB\')\n    return sha256(str(rgb.size).encode() + rgb.tobytes())\n\ndef dhash(image):\n    small = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(small.getdata())\n    result = 0\n    for y in range(8):\n        for x in range(8):\n            result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\ndef crop_box(bbox, original_size, downloaded_size):\n    x, y, width, height = bbox\n    sx, sy = downloaded_size[0]/original_size[0], downloaded_size[1]/original_size[1]\n    return (max(0, math.floor(x*sx)), max(0, math.floor(y*sy)),\n            min(downloaded_size[0], math.ceil((x+width)*sx)),\n            min(downloaded_size[1], math.ceil((y+height)*sy)))\n\ndef fetch_bytes(url, timeout=12):\n    request = urllib.request.Request(url, headers={\'User-Agent\': \'EcoSort-AI research dataset preparation\'})\n    with urllib.request.urlopen(request, timeout=timeout) as response:\n        return response.read()\n\nclass UnionFind:\n    def __init__(self, values):\n        self.parent = {v: v for v in values}\n    def find(self, value):\n        while self.parent[value] != value:\n            self.parent[value] = self.parent[self.parent[value]]\n            value = self.parent[value]\n        return value\n    def union(self, a, b):\n        a, b = self.find(a), self.find(b)\n        if a != b:\n            self.parent[max(a, b)] = min(a, b)\n\ndef grouped_splits(groups, seed=42):\n    """Input maps group IDs to crisp-packet presence; every group stays in one split."""\n    assignments = {}\n    for has_crisp in (False, True):\n        ids = sorted(k for k, flag in groups.items() if flag == has_crisp)\n        random.Random(seed + int(has_crisp)).shuffle(ids)\n        n = len(ids)\n        n_val = max(1, round(n*.15)) if n >= 7 else 0\n        n_test = n_val\n        for index, group_id in enumerate(ids):\n            assignments[group_id] = (\'val\' if index < n_val else\n                                      \'test\' if index < n_val+n_test else \'train\')\n    return assignments\n\ndef prepare_taco(output, annotation_bytes=None, workers=12, min_side=48):\n    """Download official preview images; keep only resolvable crops >= min_side pixels."""\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = annotation_bytes if annotation_bytes is not None else fetch_bytes(ANNOTATIONS_URL)\n    annotations = json.loads(raw)\n    (output/\'annotations_source.json\').write_bytes(raw)\n    categories = {c[\'id\']: c[\'name\'] for c in annotations[\'categories\']}\n    images = {im[\'id\']: im for im in annotations[\'images\']}\n    selected = [a for a in annotations[\'annotations\'] if categories[a[\'category_id\']] in CATEGORIES]\n    by_image = defaultdict(list)\n    for ann in selected:\n        by_image[ann[\'image_id\']].append(ann)\n    cache = output/\'source_cache\'\n    cache.mkdir(exist_ok=True)\n    def download(image_id):\n        info = images[image_id]\n        path = cache/f\'image_{image_id:05d}.jpg\'\n        urls = list(dict.fromkeys(u for u in (info.get(\'flickr_640_url\'), info.get(\'flickr_url\')) if u))\n        if path.exists():\n            with Image.open(path) as im:\n                image = im.convert(\'RGB\')\n            if max(image.size) > 640:\n                image = ImageOps.contain(image, (640, 640), Image.Resampling.LANCZOS)\n                image.save(path, quality=95)\n                with Image.open(path) as im:\n                    image = im.convert(\'RGB\')\n            return image_id, image, (path.with_suffix(\'.url\').read_text() if path.with_suffix(\'.url\').exists() else \'cached URL unknown\'), None\n        failures = []\n        for url in urls:\n            for attempt in range(1):\n                try:\n                    content = fetch_bytes(url)\n                    with Image.open(io.BytesIO(content)) as im:\n                        image = ImageOps.contain(im.convert(\'RGB\'), (640, 640), Image.Resampling.LANCZOS)\n                    image.save(path, quality=95)\n                    path.with_suffix(\'.url\').write_text(url)\n                    # Return the same cached decoded pixels on initial and repeat runs.\n                    with Image.open(path) as im:\n                        return image_id, im.convert(\'RGB\'), url, None\n                except Exception as exc:\n                    failures.append(type(exc).__name__ + \': \' + str(exc))\n        return image_id, None, None, failures\n    downloaded = {}\n    errors = []\n    ids = sorted(by_image)\n    print(f\'TACO: {len(selected)} annotated candidate objects in {len(ids)} source images.\', flush=True)\n    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:\n        for i, (image_id, image, url, error) in enumerate(pool.map(download, ids), 1):\n            if image is None:\n                errors.append({\'image_id\': image_id, \'errors\': error})\n            else:\n                downloaded[image_id] = (image, url)\n            if i % 25 == 0 or i == len(ids):\n                print(f\'Downloaded/checking {i}/{len(ids)} sources; failures {len(errors)}.\', flush=True)\n    # Group exact copies and near-identical whole scenes BEFORE deciding splits.\n    uf = UnionFind(sorted(downloaded))\n    fingerprints = {}\n    for image_id in sorted(downloaded):\n        image, _ = downloaded[image_id]\n        fingerprints[image_id] = (pixel_hash(image), dhash(image))\n    scene_ids = sorted(downloaded)\n    for i, a in enumerate(scene_ids):\n        for b in scene_ids[:i]:\n            if fingerprints[a][0] == fingerprints[b][0] or (fingerprints[a][1]^fingerprints[b][1]).bit_count() <= 4:\n                uf.union(a, b)\n    candidates = []\n    crop_fingerprints = {}\n    skipped = Counter()\n    for image_id in scene_ids:\n        image, url = downloaded[image_id]\n        info = images[image_id]\n        for ann in sorted(by_image[image_id], key=lambda x:x[\'id\']):\n            box = crop_box(ann[\'bbox\'], (info[\'width\'], info[\'height\']), image.size)\n            if box[2]-box[0] < min_side or box[3]-box[1] < min_side:\n                skipped[\'too_small\'] += 1\n                continue\n            crop = image.crop(box)\n            digest = pixel_hash(crop)\n            if digest in crop_fingerprints:\n                uf.union(image_id, crop_fingerprints[digest])\n                skipped[\'exact_crop_duplicate\'] += 1\n                continue\n            crop_fingerprints[digest] = image_id\n            candidates.append((image_id, ann, crop, digest, url))\n    group_flags = {}\n    for image_id, ann, crop, digest, url in candidates:\n        gid = uf.find(image_id)\n        group_flags[gid] = group_flags.get(gid, False) or categories[ann[\'category_id\']] == \'Crisp packet\'\n    split_map = grouped_splits(group_flags)\n    records = []\n    for image_id, ann, crop, digest, url in candidates:\n        split = split_map[uf.find(image_id)]\n        relative = f\'{split}/plastic/taco_{image_id:05d}_{ann["id"]:05d}.png\'\n        path = output/relative\n        path.parent.mkdir(parents=True, exist_ok=True)\n        crop.save(path)\n        records.append({\'file\':relative,\'image_id\':image_id,\'annotation_id\':ann[\'id\'],\n            \'group_id\':uf.find(image_id),\'split\':split,\'label\':\'plastic\',\n            \'source_category\':categories[ann[\'category_id\']], \'bbox\':ann[\'bbox\'],\n            \'downloaded_crop_size\':list(crop.size),\'pixel_sha256\':digest,\n            \'download_url\':url,\'original_url\':images[image_id].get(\'flickr_url\'),\n            \'license_id\':images[image_id].get(\'license\')})\n    counts = Counter(r[\'split\'] for r in records)\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    for split in (\'train\',\'val\',\'test\'):\n        if counts[split] < 15:\n            raise RuntimeError(f\'Too few usable TACO {split} crops ({counts[split]}). No training started. Review download_errors.json and source availability.\')\n    manifest = {\'taco_commit\':TACO_COMMIT,\'annotations_url\':ANNOTATIONS_URL,\n        \'annotations_sha256\':sha256(raw),\'candidate_objects\':len(selected),\n        \'downloaded_source_images\':len(downloaded),\'failed_source_images\':len(errors),\n        \'counts\':dict(counts),\'source_category_counts\':dict(Counter(r[\'source_category\'] for r in records)),\n        \'skip_counts\':dict(skipped),\'min_crop_side\':min_side,\'seed\':42,\n        \'split_unit\':\'source scene / exact crop duplicate / whole-image dHash distance <= 4\',\n        \'label_policy\':POLICY, \'licenses\':annotations.get(\'licenses\',[]),\'records\':records}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'Usable new TACO crops:\', dict(counts), \'Total:\', len(records), flush=True)\n    # Show only training examples; leave test images and scores untouched until final evaluation.\n    sample=random.Random(42).sample([r for r in records if r[\'split\']==\'train\'],min(30,counts[\'train\']))\n    sheet=Image.new(\'RGB\',(900,6*145),\'white\')\n    draw=ImageDraw.Draw(sheet)\n    for i,r in enumerate(sample):\n        with Image.open(output/r[\'file\']) as im:\n            preview=ImageOps.contain(im,(170,115))\n        x,y=(i%5)*180,(i//5)*145\n        sheet.paste(preview,(x,y))\n        draw.text((x,y+117),f\'{r["annotation_id"]} {r["source_category"][:19]}\',fill=\'black\')\n    sheet.save(output/\'training_preview.jpg\')\n    return manifest\n\nif __name__ == \'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args()\n    prepare_taco(args.output)\n', 'train_taco': '"""Train a fresh six-class classifier with TACO plastic-packaging crops."""\nimport csv\nimport hashlib\nimport json\nimport random\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image\nfrom taco_data import CLASSES, POLICY, pixel_hash\n\n\ndef read_trashnet(archive, destination):\n    """Extract only class images. Never overwrite or modify the uploaded archive."""\n    destination = Path(destination)\n    records = []\n    with zipfile.ZipFile(archive) as z:\n        members = []\n        for member in z.infolist():\n            parts = Path(member.filename.replace(\'\\\\\', \'/\')).parts\n            for i in range(len(parts)-2):\n                if (parts[i] in (\'train\',\'val\',\'test\') and parts[i+1] in CLASSES\n                        and len(parts[i:]) == 3 and Path(parts[-1]).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')):\n                    members.append((member, parts[i], parts[i+1], parts[-1]))\n                    break\n        if not members:\n            raise ValueError(\'Upload EcoSort_TrashNet_split.zip containing train/val/test class folders.\')\n        seen_paths = set()\n        for member, split, label, name in members:\n            path = destination/split/label/name\n            if str(path) in seen_paths:\n                raise ValueError(f\'Duplicate ZIP destination: {split}/{label}/{name}\')\n            seen_paths.add(str(path))\n            path.parent.mkdir(parents=True,exist_ok=True)\n            path.write_bytes(z.read(member))\n            with Image.open(path) as im:\n                digest = pixel_hash(im)\n            records.append({\'file\':str(path),\'split\':split,\'label\':label,\'source\':\'trashnet\',\'pixel_sha256\':digest})\n    for split in (\'train\',\'val\',\'test\'):\n        labels = {r[\'label\'] for r in records if r[\'split\']==split}\n        if labels != set(CLASSES):\n            raise ValueError(f\'TrashNet {split} is missing classes: {set(CLASSES)-labels}\')\n    # Remove EVERY member of groups crossing a split or label, rather than choosing a label by guesswork.\n    by_hash = defaultdict(list)\n    for r in records:\n        by_hash[r[\'pixel_sha256\']].append(r)\n    keep, excluded = [], []\n    for group in by_hash.values():\n        conflict = len({(r[\'split\'],r[\'label\']) for r in group}) > 1\n        if conflict:\n            excluded.extend(dict(r,reason=\'duplicate crossing splits or labels\') for r in group)\n        else:\n            keep.append(group[0])\n            excluded.extend(dict(r,reason=\'duplicate within class and split\') for r in group[1:])\n    return keep, excluded\n\n\ndef prepare_records(archive, taco_root, work, data_root=None):\n    work = Path(work)\n    work.mkdir(parents=True,exist_ok=True)\n    original, excluded = read_trashnet(archive,Path(data_root) if data_root else work/\'trashnet\')\n    taco_root = Path(taco_root)\n    manifest = json.loads((taco_root/\'manifest.json\').read_text())\n    original_hashes = {r[\'pixel_sha256\'] for r in original}\n    blocked = {r[\'group_id\'] for r in manifest[\'records\'] if r[\'pixel_sha256\'] in original_hashes}\n    new = []\n    groups = defaultdict(set)\n    for r in manifest[\'records\']:\n        groups[r[\'group_id\']].add(r[\'split\'])\n        row = dict(r,file=str(taco_root/r[\'file\']),source=\'taco\')\n        if r[\'group_id\'] in blocked:\n            excluded.append(dict(row,reason=\'TACO group matches TrashNet decoded pixels\'))\n        else:\n            new.append(row)\n    if any(len(splits)>1 for splits in groups.values()):\n        raise ValueError(\'TACO source group crosses splits; training stopped.\')\n    counts = Counter(r[\'split\'] for r in new)\n    if counts[\'train\']<100 or counts[\'val\']<15 or counts[\'test\']<15:\n        raise ValueError(f\'Not enough NEW wrapper images to train: {dict(counts)}\')\n    all_records = original+new\n    split_hashes = defaultdict(set)\n    for r in all_records:\n        split_hashes[r[\'pixel_sha256\']].add(r[\'split\'])\n    if any(len(s)>1 for s in split_hashes.values()):\n        raise ValueError(\'Exact image leakage detected between splits.\')\n    (work/\'excluded_images.json\').write_text(json.dumps(excluded,indent=2))\n    (work/\'dataset_records.json\').write_text(json.dumps(all_records,indent=2))\n    with open(archive, \'rb\') as handle:\n        archive_digest = hashlib.file_digest(handle, \'sha256\').hexdigest()\n    summary = {\'archive_name\':Path(archive).name,\'archive_sha256\':archive_digest,\n        \'trashnet_counts\':dict(Counter(r[\'split\'] for r in original)),\n        \'new_taco_counts\':dict(counts),\'excluded_count\':len(excluded),\'classes\':CLASSES,\n        \'class_counts_by_split\':{s:dict(Counter(r[\'label\'] for r in all_records if r[\'split\']==s)) for s in (\'train\',\'val\',\'test\')}}\n    (work/\'dataset_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(\'TrashNet after duplicate exclusion:\',dict(Counter(r[\'split\'] for r in original)),flush=True)\n    print(\'NEW wrapper images:\',dict(counts),flush=True)\n    return all_records\n\n\ndef train(records, output, head_epochs=3, finetune_epochs=7, batch_size=32, device_name="cuda"):\n    import numpy as np\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    if device_name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training. No training started.\')\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    device=torch.device(device_name)\n    mean,std=[.485,.456,.406],[.229,.224,.225]\n    evaluation=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    augmentation=transforms.Compose([transforms.RandomResizedCrop(224,scale=(.75,1.)),transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    class Images(Dataset):\n        def __init__(self, rows, transform): self.rows,self.transform=rows,transform\n        def __len__(self): return len(self.rows)\n        def __getitem__(self,index):\n            r=self.rows[index]\n            with Image.open(r[\'file\']) as im: image=self.transform(im.convert(\'RGB\'))\n            return image,CLASSES.index(r[\'label\'])\n    split_rows={s:[r for r in records if r[\'split\']==s] for s in (\'train\',\'val\',\'test\')}\n    generator=torch.Generator().manual_seed(42)\n    loaders={s:DataLoader(Images(rows,augmentation if s==\'train\' else evaluation),batch_size=batch_size,\n        shuffle=s==\'train\',num_workers=0,generator=generator if s==\'train\' else None) for s,rows in split_rows.items()}\n    print(\'Device:\',torch.cuda.get_device_name(0) if device_name==\'cuda\' else device_name,flush=True)\n    print(\'Starting a NEW ImageNet-initialized ResNet18. No previous EcoSort weights loaded.\',flush=True)\n    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\n    model.fc=nn.Linear(model.fc.in_features,len(CLASSES))\n    model.to(device)\n    counts=Counter(r[\'label\'] for r in split_rows[\'train\'])\n    weights=torch.tensor([len(split_rows[\'train\'])/(len(CLASSES)*counts[c]) for c in CLASSES],device=device)\n    loss_fn=nn.CrossEntropyLoss(weight=weights)\n    def predict(rows):\n        model.eval()\n        truth,prediction=[],[]\n        with torch.inference_mode():\n            for x,y in DataLoader(Images(rows,evaluation),batch_size=batch_size,num_workers=0):\n                p=model(x.to(device)).argmax(1).cpu().tolist()\n                prediction.extend(p)\n                truth.extend(y.tolist())\n        return truth,prediction\n    def metrics(rows):\n        y,p=predict(rows)\n        result={\'n\':len(y),\'accuracy\':accuracy_score(y,p),\n            \'confusion_matrix\':confusion_matrix(y,p,labels=list(range(6))).tolist()}\n        if {r[\'label\'] for r in rows}=={\'plastic\'}:\n            result[\'plastic_recall\']=result[\'accuracy\']\n            result[\'note\']=\'Positive-only wrapper set: this measures recall, not six-class accuracy or plastic precision.\'\n            result[\'by_source_category\']={c:{\'n\':sum(r.get(\'source_category\')==c for r in rows),\n                \'plastic_recall\':sum(pp==4 for r,pp in zip(rows,p) if r.get(\'source_category\')==c)/sum(r.get(\'source_category\')==c for r in rows)}\n                for c in sorted({r[\'source_category\'] for r in rows})}\n        else:\n            result[\'macro_f1\']=f1_score(y,p,labels=list(range(6)),average=\'macro\',zero_division=0)\n            result[\'class_report\']=classification_report(y,p,labels=list(range(6)),target_names=CLASSES,output_dict=True,zero_division=0)\n        return result\n    history=[]\n    best=(-1.,-1.)\n    checkpoint=output/\'ecosort_resnet18_taco_plastic.pth\'\n    epoch=0\n    for stage,n_epochs in [(\'head\',head_epochs),(\'layer4\',finetune_epochs)]:\n        if stage==\'layer4\':\n            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n        for name,param in model.named_parameters():\n            param.requires_grad=name.startswith(\'fc.\') or (stage==\'layer4\' and name.startswith(\'layer4.\'))\n        groups=[{\'params\':model.fc.parameters(),\'lr\':1e-3 if stage==\'head\' else 1e-4}]\n        if stage==\'layer4\': groups.append({\'params\':model.layer4.parameters(),\'lr\':1e-5})\n        optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n        for _ in range(n_epochs):\n            epoch+=1\n            # Frozen layers and their batch-normalization statistics remain frozen.\n            model.eval()\n            model.fc.train()\n            if stage==\'layer4\': model.layer4.train()\n            total_loss,correct,n=0.,0,0\n            for x,y in loaders[\'train\']:\n                x,y=x.to(device),y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits=model(x)\n                loss=loss_fn(logits,y)\n                loss.backward()\n                optimizer.step()\n                total_loss+=loss.item()*len(y)\n                correct+=(logits.argmax(1)==y).sum().item()\n                n+=len(y)\n            val=metrics(split_rows[\'val\'])\n            wrapper_val=metrics([r for r in split_rows[\'val\'] if r[\'source\']==\'taco\'])\n            row={\'epoch\':epoch,\'stage\':stage,\'train_loss\':total_loss/n,\'train_accuracy\':correct/n,\n                \'val_accuracy\':val[\'accuracy\'],\'val_macro_f1\':val[\'macro_f1\'],\'wrapper_val_recall\':wrapper_val[\'plastic_recall\']}\n            history.append(row)\n            print(f"Epoch {epoch}/{head_epochs+finetune_epochs}: val F1 {val[\'macro_f1\']:.3f}, wrapper val recall {wrapper_val[\'plastic_recall\']:.1%}",flush=True)\n            score=(val[\'macro_f1\'],val[\'accuracy\'])\n            if score>best:\n                best=score\n                torch.save({\'model_state_dict\':model.state_dict(),\'classes\':CLASSES,\'class_names\':CLASSES,\n                    \'epoch\':epoch,\'val_macro_f1\':best[0],\'image_size\':224,\'mean\':mean,\'std\':std,\n                    \'label_policy\':POLICY,\'initialization\':\'ImageNet1K V1; no previous EcoSort checkpoint\'},checkpoint)\n    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    final={\'classes\':CLASSES,\'label_policy\':POLICY,\'selection\':\'highest combined validation macro F1, accuracy breaks ties\',\n        \'selected_epoch\':torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'epoch\'],\n        \'trashnet_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'trashnet\']),\n        \'new_wrapper_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'taco\']),\n        \'combined_test\':metrics(split_rows[\'test\']),\n        \'limitations\':[\'TrashNet test has been inspected in earlier experiments; interpret as diagnostic.\',\n            \'TACO holdout is grouped by source and near-identical scenes, not independently collected deployment photos.\',\n            \'Packaging categories are project labels, not verified material composition or recycling advice.\',\n            \'TACO crops differ from whole-image TrashNet photos; evaluate fresh user photos before deployment.\']}\n    (output/\'metrics.json\').write_text(json.dumps(final,indent=2))\n    with (output/\'training_history.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n    print(\'Selected epoch:\',final[\'selected_epoch\'],flush=True)\n    print(\'TrashNet test accuracy:\',f"{final[\'trashnet_test\'][\'accuracy\']:.1%}",flush=True)\n    print(\'NEW wrapper test recall:\',f"{final[\'new_wrapper_test\'][\'plastic_recall\']:.1%}",flush=True)\n    return final\n'}
for name,text in MODULES.items(): (source/f'{name}.py').write_text(text)
sys.path.insert(0,str(source))
# Reload the embedded source if this cell is run again.
for name in MODULES: sys.modules.pop(name,None)
from taco_data import prepare_taco
from train_taco import prepare_records, train
result=work/'Result'
result.mkdir(exist_ok=True)
(result/'Src').mkdir(exist_ok=True)
(result/"Src/predict_taco.py").write_text('"""Prediction using the new TACO checkpoint; keeps the earlier predictor unchanged."""\nimport argparse\nfrom pathlib import Path\nimport torch\nfrom PIL import Image\nfrom torchvision import models, transforms\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'image\')\n    parser.add_argument(\'--model\',default=str(Path(__file__).resolve().parent.parent/\'Models\'/\'ecosort_resnet18_taco_plastic.pth\'))\n    args=parser.parse_args()\n    checkpoint=torch.load(args.model,map_location=\'cpu\',weights_only=True)\n    classes=checkpoint[\'class_names\']\n    model=models.resnet18(weights=None)\n    model.fc=torch.nn.Linear(model.fc.in_features,len(classes))\n    model.load_state_dict(checkpoint[\'model_state_dict\'])\n    model.eval()\n    transform=transforms.Compose([transforms.Resize((checkpoint[\'image_size\'],checkpoint[\'image_size\'])),\n        transforms.ToTensor(),transforms.Normalize(checkpoint[\'mean\'],checkpoint[\'std\'])])\n    with Image.open(args.image) as image:\n        x=transform(image.convert(\'RGB\')).unsqueeze(0)\n    with torch.inference_mode(): scores=model(x).softmax(1)[0]\n    for i in scores.argsort(descending=True).tolist(): print(f\'{classes[i]}: {scores[i]:.1%}\')\n    print(\'Scores are model outputs, not calibrated probabilities of correctness.\')\n\n\nif __name__==\'__main__\': main()\n')
(result/"README.md").write_text('# Train EcoSort on new plastic packaging\n\nOpen `NoteBook/EcoSort_NEW_TACO_Plastic.ipynb` in Google Colab, select a T4 GPU, and run the single code cell. Upload **EcoSort_TrashNet_split.zip**, containing the existing six-class train/val/test folders. Do not upload an old `.pth` model. Keep the notebook open while the runtime works. Download the output ZIP when prompted.\n\nThe notebook contains the preparation and training source, so it can run without GitHub credentials. It automatically downloads TACO annotations and source images, extracts labelled plastic-packaging objects, combines them with TrashNet, and trains a new ResNet18 from ImageNet weights. It does not use the three hand-picked photos or earlier wrapper-labelled checkpoints. Default training: 3 head epochs, then 7 epochs updating layer4 and the head. The six output classes remain cardboard, glass, metal, paper, plastic, trash.\n\n## Labels and source\n\nSource: https://github.com/pedropro/TACO, pinned annotation commit `70479ad9adc3da971a8ae75ac97eb1026910eec7`.\n\nThe official annotations contain 451 Plastic film, 260 Other plastic wrapper and 39 Crisp packet objects (750 candidates, not 750 training images). Only successfully downloaded, unique crops with both sides at least 48 pixels enter the experiment. Actual counts and failed downloads are printed and saved. The workflow refuses to train with fewer than 100 new training examples or fewer than 15 validation or test examples.\n\n**Project convention:** these three packaging categories map to plastic, including metallized snack packets. This is an appearance/packaging label, not verified chemical composition or recycling guidance. The broader category counts do not mean hundreds of snack-packet photographs.\n\nImages are downloaded from the URLs provided by TACO. Source URLs, bounding boxes, annotation IDs, category names, image-license metadata and annotation hash are preserved in the manifest. A missing image license remains missing; the toolkit\'s MIT license should not be assumed to license every photograph. Review original source permissions before redistributing image collections. No images are committed to this repository.\n\n## Splits and measurement\n\n- All crops from the same source image stay in the same split. Exact duplicate crops and whole scenes within dHash distance 4 are grouped before splitting. This is a heuristic, not a guarantee that every related scene is detected.\n- Source groups are split approximately 70/15/15 with seed 42, stratified on presence of crisp packets when enough groups exist.\n- TrashNet copies crossing splits or class labels are all excluded from this experiment; other identical copies within one class/split are reduced to one. Uploaded originals are preserved. Exact decoded-pixel collisions with TrashNet exclude the whole matching TACO source group.\n- Selection uses combined validation macro F1, with accuracy breaking ties. Test inference happens only after selecting the final checkpoint.\n- Report TrashNet six-class test metrics and new-wrapper plastic recall separately. The TACO subset contains only plastic-labelled examples, so it cannot establish plastic precision or six-class accuracy by itself.\n- Earlier EcoSort experiments already examined the TrashNet test set. Its results are diagnostic rather than an untouched final benchmark. TACO holdout results also do not establish performance on independently collected local packet photographs.\n- TACO object crops have different backgrounds/framing from TrashNet whole images. Check fresh, unused real photos before claiming an improvement.\n\n## Outputs\n\n`EcoSort_TACO_Plastic_Result.zip` includes `Models/ecosort_resnet18_taco_plastic.pth`, metrics, training history, dataset records, excluded-image list, TACO provenance/download errors, the training-only preview, a compatible prediction script, documentation and the Colab package versions. It does not include the downloaded image collection. Keep all output files together for reproducibility. Output names differ from earlier models.\n\nAfter extracting the ZIP into your project, predict in the inference environment with:\n\n```bat\npython Src\\predict_taco.py "C:\\path\\to\\new-photo.jpg"\n```\n\nReproduction outside Colab requires Python, PyTorch/torchvision, Pillow, NumPy and scikit-learn with a CUDA GPU for the training function. Exact versions are captured at run time, rather than assuming the Windows CPU package list matches Colab.\n\n## Dataset preparation check (24 October 2026)\n\nAll 475 selected source images were downloaded. Filtering left **309 new crops: 218 train, 45 validation, 46 test**. Categories: 186 plastic film, 96 other wrappers and 27 crisp packets. Another 441 candidate crops were too small. Source-group and exact-crop overlap checks passed. These are preparation counts, not trained accuracy. Availability changes may affect later downloads.\n\n## Validation of the workflow\n\nSee the pull request for checks actually run. Preparing a notebook or passing a synthetic CPU smoke test does not mean the real dataset has been GPU-trained. Real training scores are produced only by running the Colab notebook.\n\nPlease cite: Pedro F. Proença and Pedro Simões, *TACO: Trash Annotations in Context for Litter Detection*, 2020, https://arxiv.org/abs/2003.06975.\n')
versions={n:importlib.metadata.version(n) for n in ['torch','torchvision','Pillow','numpy','scikit-learn']}
(result/'package_versions.json').write_text(json.dumps(versions,indent=2))
try:
    print("Downloading NEW TACO packaging data. This stage can take several minutes.",flush=True)
    manifest=prepare_taco(work/'taco')
    display(DisplayImage(filename=str(work/'taco/training_preview.jpg')))
    records=prepare_records(archive,work/'taco',result/'Results',data_root=work/'trashnet')
    for name in ['manifest.json','download_errors.json','annotations_source.json','training_preview.jpg']:
        shutil.copy2(work/'taco'/name,result/'Results'/f'taco_{name}')
    train(records,result/'Models')
    # Metrics/history live alongside the checkpoint in Models.
    print("Finished. Download and preserve the full result ZIP.",flush=True)
except Exception:
    (result/'error.txt').write_text(traceback.format_exc())
    debug=shutil.make_archive('/content/EcoSort_TACO_Debug','zip',result)
    print("Stopped safely. Download the debug ZIP if offered; training may not have completed.")
    files.download(debug)
    raise
bundle=shutil.make_archive('/content/EcoSort_TACO_Plastic_Result','zip',result)
files.download(bundle)
